<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.3-rag-engine/practice/GCP_Capstone_4.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 4.3 — Vertex AI RAG Engine

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install the SDKs, authenticate with Application Default Credentials, and initialize the Vertex RAG module plus the `google-genai` client. Run this cell first — every exercise below depends on `rag`, `client`, `types`, and `PROJECT_ID`.

In [ ]:
!pip install -q google-cloud-aiplatform google-genai
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'us-central1'           # course region (asia-south1 for India prod)

from vertexai import rag
import vertexai
from google import genai
from google.genai import types

vertexai.init(project=PROJECT_ID, location=LOCATION)
client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # Gemini 3.x generation: global (corpus/LOCATION stays regional)
print('Setup complete.')

## Enable Serverless mode (one-time, required on new projects)
Vertex AI RAG Engine's default corpus store ("Scaled"/Spanner mode) is **allowlist-only for new projects** in `us-central1`, `us-east1`, and `us-east4`, so a fresh project fails `create_corpus` with `INVALID_ARGUMENT: ... Spanner mode ... restricted to only allowlisted projects`. **Serverless mode** is open to everyone with no allowlist. The switch below is **project-level and one-time** (idempotent — safe to re-run); afterwards `create_corpus()` uses the serverless vector store automatically and every cell below works unchanged.

In [ ]:
# One-time, project-level: put Vertex AI RAG Engine in Serverless mode so create_corpus()
# works on new projects without allowlisting. Idempotent — safe to re-run.
import google.auth, google.auth.transport.requests, requests
_RAG_LOCATION = 'us-central1'   # RAG Engine region (the corpus stays regional)
_creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
_creds.refresh(google.auth.transport.requests.Request())
_r = requests.patch(
    f'https://{_RAG_LOCATION}-aiplatform.googleapis.com/v1beta1/'
    f'projects/{PROJECT_ID}/locations/{_RAG_LOCATION}/ragEngineConfig',
    headers={'Authorization': f'Bearer {_creds.token}'},
    json={'ragManagedDbConfig': {'serverless': {}}}, timeout=60)
if _r.ok:
    print('RAG Engine: Serverless mode ready.')
else:
    print(f'Could not set Serverless mode ({_r.status_code}): {_r.text[:200]}')
    print('Fallback: Console > RAG Engine > Switch to Serverless, or use a non-restricted region.')

## Exercise 1: Create a Corpus

**Difficulty:** Easy

Create a RAG corpus with text-embedding-005. Print the resource name.

1. Configure RagEmbeddingModelConfig
2. Call rag.create_corpus()
3. Print corpus.name

In [ ]:
embedding_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model='publishers/google/models/text-embedding-005'
    )
)

corpus = rag.create_corpus(
    display_name='documind-lesson43',
    description='Lesson 4.3 test corpus',
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_config),
)
print(f'Corpus: {corpus.name}')

# List corpora
for c in rag.list_corpora():
    print(f'  {c.display_name}: {c.name}')

## Exercise 2: Import from GCS

**Difficulty:** Easy

Upload a PDF to GCS. Import into corpus. Print imported file count.

1. gsutil cp file.pdf gs://bucket/docs/
2. rag.import_files() with chunk_size=512
3. Check imported_rag_files_count

In [ ]:
%%bash
# Upload your test document(s) to GCS first. Replace YOUR-BUCKET.
gsutil cp test.pdf gs://YOUR-BUCKET/docs/
gsutil ls gs://YOUR-BUCKET/docs/

In [ ]:
response = rag.import_files(
    corpus.name,
    ['gs://YOUR-BUCKET/docs/'],  # CHANGE
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512, chunk_overlap=100)),
    max_embedding_requests_per_min=900,
)
print(f'Imported: {response.imported_rag_files_count}')
print(f'Skipped: {response.skipped_rag_files_count}')

## Exercise 3: Direct Retrieval

**Difficulty:** Easy

Use retrieval_query() to search corpus. Print top-5 chunks with scores.

1. Call rag.retrieval_query()
2. Loop response.contexts.contexts
3. Print source_uri, score, text preview

In [ ]:
response = rag.retrieval_query(
    rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
    text='What is RAG?',
    rag_retrieval_config=rag.RagRetrievalConfig(
        top_k=5,
        filter=rag.Filter(vector_distance_threshold=0.5)),
)

for ctx in response.contexts.contexts:
    print(f'Source: {ctx.source_uri}')
    print(f'Score: {ctx.score:.3f}, Distance: {ctx.distance:.3f}')
    print(f'Text: {ctx.text[:150]}...\n')

## Exercise 4: Grounded Generation

**Difficulty:** Medium

Use types.Tool(retrieval=...) + generate_content(). Print answer + grounding citations.

1. Create RAG tool from corpus
2. Pass to GenerativeModel
3. Print grounding_chunks and grounding_supports

In [ ]:
rag_retrieval_tool = types.Tool(
    retrieval=types.Retrieval(
        vertex_rag_store=types.VertexRagStore(
            rag_resources=[types.VertexRagStoreRagResource(rag_corpus=corpus.name)],
            rag_retrieval_config=types.RagRetrievalConfig(
                top_k=5,
                filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))

response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Summarize the main topics in my documents',
    config=types.GenerateContentConfig(tools=[rag_retrieval_tool]))
print(response.text)

# Grounding metadata — automatic citations
for candidate in response.candidates:
    gm = candidate.grounding_metadata
    if gm and gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            print(f'  Source: {chunk.retrieved_context.uri}')
    if gm and gm.grounding_supports:
        for support in gm.grounding_supports:
            print(f'  Claim: {support.segment.text}')
            print(f'  Backed by: {support.grounding_chunk_indices}')

## Exercise 5: Drive Connector

**Difficulty:** Medium

Share a Drive folder with the RAG service account. Import and query.

1. Find RAG service account in IAM
2. Share Drive folder as Viewer
3. Import with Drive URL path

In [ ]:
%%bash
# Step 1: Find the RAG service account to share your Drive folder with.
# The RAG Engine uses the Vertex AI RAG data service agent.
# NOTE: %%bash runs a subshell — the Python PROJECT_ID is NOT visible here,
# so resolve the active project from gcloud config instead.
PROJECT_ID=$(gcloud config get-value project 2>/dev/null)
gcloud projects get-iam-policy "$PROJECT_ID" \
  --flatten='bindings[].members' \
  --format='table(bindings.members)' \
  --filter='bindings.members:rag' 2>/dev/null || true

# Typical form:
#   service-<PROJECT_NUMBER>@gcp-sa-vertex-rag.iam.gserviceaccount.com
# Step 2 (manual): In Google Drive, share the target folder as Viewer
#   with that service account email.

In [ ]:
# Step 3: Import from the shared Drive folder, then retrieve.
drive_response = rag.import_files(
    corpus.name,
    ['https://drive.google.com/drive/folders/YOUR_FOLDER_ID'],  # CHANGE
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100)),
    max_embedding_requests_per_min=900,
)
print(f'Drive import: {drive_response.imported_rag_files_count}')

# Confirm Drive-sourced chunks come back
drive_hits = rag.retrieval_query(
    rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
    text='What do the Drive documents cover?',
    rag_retrieval_config=rag.RagRetrievalConfig(top_k=5),
)
for ctx in drive_hits.contexts.contexts:
    print(f'  {ctx.source_uri}  (score {ctx.score:.3f})')

## Exercise 6: Layout Parser Import

**Difficulty:** Medium

Import PDFs with LayoutParserConfig. Compare chunk quality vs default.

1. Pass layout_parser=rag.LayoutParserConfig(...)
2. Query same question with and without Layout Parser
3. Compare chunk coherence

In [ ]:
# PREREQUISITE: this exercise needs your OWN GCS bucket of docs and a Document AI
# Layout-Parser processor id (replace YOUR-BUCKET / YOUR_LAYOUT_PROCESSOR_ID below).
# Layout-aware parsing keeps headings, tables and paragraphs intact instead of
# splitting on raw token counts. Import the SAME source into a fresh corpus so
# you can compare chunk coherence side by side.

layout_corpus = rag.create_corpus(
    display_name='documind-lesson43-layout',
    description='Layout Parser comparison corpus',
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_config),
)
print(f'Layout corpus: {layout_corpus.name}')

rag.import_files(
    layout_corpus.name,
    ['gs://YOUR-BUCKET/docs/'],  # CHANGE — same source as Exercise 2
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100)),
    layout_parser=rag.LayoutParserConfig(
        processor_name=f'projects/{PROJECT_ID}/locations/us/processors/YOUR_LAYOUT_PROCESSOR_ID',
        max_parsing_requests_per_min=120,
    ),
    max_embedding_requests_per_min=900,
)

question = 'What is RAG?'
print('--- Default chunking ---')
for ctx in rag.retrieval_query(
        rag_resources=[rag.RagResource(rag_corpus=corpus.name)],
        text=question,
        rag_retrieval_config=rag.RagRetrievalConfig(top_k=3)).contexts.contexts:
    print(f'  {ctx.text[:180]}...')

print('\n--- Layout Parser chunking ---')
for ctx in rag.retrieval_query(
        rag_resources=[rag.RagResource(rag_corpus=layout_corpus.name)],
        text=question,
        rag_retrieval_config=rag.RagRetrievalConfig(top_k=3)).contexts.contexts:
    print(f'  {ctx.text[:180]}...')

## Exercise 7: Multi-Corpus Search

**Difficulty:** Challenge

Create 2 corpora (engineering + product). Import different docs. Query across both.

1. Create corpus_eng and corpus_prod
2. Import different document sets into each
3. Pass both as RagResource objects in one query

In [ ]:
# Create two topic-specific corpora sharing the same embedding config.
corpus_eng = rag.create_corpus(
    display_name='documind-eng',
    description='Engineering docs',
    backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=embedding_config))
corpus_prod = rag.create_corpus(
    display_name='documind-product',
    description='Product docs',
    backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=embedding_config))
print(f'Eng:     {corpus_eng.name}')
print(f'Product: {corpus_prod.name}')

tx = rag.TransformationConfig(
    chunking_config=rag.ChunkingConfig(chunk_size=512, chunk_overlap=100))

rag.import_files(corpus_eng.name, ['gs://YOUR-BUCKET/engineering/'],   # CHANGE
                 transformation_config=tx, max_embedding_requests_per_min=900)
rag.import_files(corpus_prod.name, ['gs://YOUR-BUCKET/product/'],       # CHANGE
                 transformation_config=tx, max_embedding_requests_per_min=900)

# retrieval_query supports only ONE corpus per call, so query each corpus
# separately and merge the contexts by distance (closest first).
merged = []
for rc in (corpus_eng, corpus_prod):
    resp = rag.retrieval_query(
        rag_resources=[rag.RagResource(rag_corpus=rc.name)],
        text='How does the product handle document ingestion?',
        rag_retrieval_config=rag.RagRetrievalConfig(
            top_k=6, filter=rag.Filter(vector_distance_threshold=0.5)))
    merged.extend(resp.contexts.contexts)
merged.sort(key=lambda c: c.distance)
for ctx in merged[:6]:
    print(f'  {ctx.source_uri}  (score {ctx.score:.3f})')

## Exercise 8: ManagedRAG Module

**Difficulty:** Challenge

Build complete ManagedRAG class with create_corpus(), ingest(), retrieve(), ask().

1. Implement all 4 methods
2. Test end-to-end: create → ingest → ask
3. Print grounding metadata from ask()

In [ ]:
class ManagedRAG:
    def __init__(self, project, location='us-central1'):
        vertexai.init(project=project, location=location)
        self.client = genai.Client(enterprise=True, project=project, location='global')  # generation: global (corpus stays regional)
        self.corpus = None

    def create_corpus(self, name, description=''):
        emb = rag.RagEmbeddingModelConfig(
            vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
                publisher_model='publishers/google/models/text-embedding-005'))
        self.corpus = rag.create_corpus(
            display_name=name, description=description,
            backend_config=rag.RagVectorDbConfig(rag_embedding_model_config=emb))
        return self.corpus.name

    def use_corpus(self, corpus_name):
        self.corpus = rag.get_corpus(name=corpus_name)

    def ingest(self, paths, chunk_size=512, chunk_overlap=100):
        return rag.import_files(
            self.corpus.name, paths,
            transformation_config=rag.TransformationConfig(
                rag.ChunkingConfig(chunk_size=chunk_size, chunk_overlap=chunk_overlap)),
            max_embedding_requests_per_min=900)

    def retrieve(self, query, top_k=5):
        return rag.retrieval_query(
            rag_resources=[rag.RagResource(rag_corpus=self.corpus.name)],
            text=query,
            rag_retrieval_config=rag.RagRetrievalConfig(
                top_k=top_k, filter=rag.Filter(vector_distance_threshold=0.5)))

    def ask(self, question, model_name='gemini-3.6-flash'):
        rag_tool = types.Tool(retrieval=types.Retrieval(
            vertex_rag_store=types.VertexRagStore(
                rag_resources=[types.VertexRagStoreRagResource(rag_corpus=self.corpus.name)],
                rag_retrieval_config=types.RagRetrievalConfig(
                    top_k=5, filter=types.RagRetrievalConfigFilter(vector_distance_threshold=0.5)))))
        return self.client.models.generate_content(
            model=model_name, contents=question,
            config=types.GenerateContentConfig(tools=[rag_tool]))

print('ManagedRAG class ready')

In [ ]:
# End-to-end test: create -> ingest -> ask, then print grounding metadata.
mrag = ManagedRAG(PROJECT_ID, LOCATION)
mrag.create_corpus('documind-managed-e2e', 'ManagedRAG end-to-end test')
mrag.ingest(['gs://YOUR-BUCKET/docs/'])  # CHANGE

answer = mrag.ask('What are the key points in my documents?')
print(answer.text)

for candidate in answer.candidates:
    gm = candidate.grounding_metadata
    if gm and gm.grounding_chunks:
        print('\nCitations:')
        for chunk in gm.grounding_chunks:
            print(f'  Source: {chunk.retrieved_context.uri}')

## Cleanup (optional)

Delete the corpora you created to avoid ongoing storage costs. RAG Engine storage is billed per GB-month; leaving idle corpora around adds up. Uncomment to run.

In [ ]:
# for c in [corpus, layout_corpus, corpus_eng, corpus_prod, mrag.corpus]:
#     try:
#         rag.delete_corpus(name=c.name)
#         print(f'Deleted {c.name}')
#     except Exception as e:
#         print(f'Skip {c}: {e}')